# Getting Data from Wikipedia
 
This notebook shows initially how to work with the **mwclient** library to exlore the content of Wikipedia pages. It's an example of a API wrapper, that facilitates working with the underlying API, the [MediaWikiAPI](https://www.mediawiki.org/wiki/API:Main_page).

Then, we introduce a similar wrapper library, **mwviews**, to get the page views for an article. We then combine data that we can get from these two libraries, so that you can explore the concept of "anticipated hits" vs. "sleeper hits" 

**Table of Content**
1. [Installing the `mwclient` module](#sec1)
2. [Connecting to a site and working with pages](#sec2)
3. [The Wellesley College page](#sec3)
4. [Exploring Wellesley's Wiki page revisions](#sec4)
5. [Accessing Pageviews for Wiki pages](#sec5)

## 1. Installing the `mwclient` module

In this notebook I've used the module [mwclient](http://mwclient.readthedocs.io/en/latest/index.html). The name stands for "Media Wiki Client". It's a library to access the Wikipedia pages through Python.

Since this is your first time using this library, you have to install it first.  

In [ ]:
pip install mwclient

In [ ]:
# check if module is installed
import mwclient

<a id="sec2"></a>

## 2. Connecting to a wiki site and getting pages

There are many wiki websites that are accessed by MediaWikiAPI. We need to provide the URL of the one we will work with, in our case the English Wikipedia.

In [ ]:
from mwclient import Site
site = Site('en.wikipedia.org')

It's possible to search for pages based on a simple query term, given that we will search within the `site`:

In [ ]:
page = site.pages['Wellesley']
page

And we can read the text of the page, which in this case appears to be a disambiguation page with links to many pages that contain the word "Wellesley":

In [ ]:
page.text()

### 2.a What else does a page contain?

Let's looks at some properties that the page contains:

**Categories:** Most pages in Wikipedia are assigned categories, which we can access:

In [ ]:
for cat in page.categories():
    print(cat)

**Links:** A page has many links to other Wikipedia pages, we can access them too:

In [ ]:
links = [l for l in page.links()]
print(f"Total links: {len(links)}")

In [ ]:
links[:5]

**IMPORTANT - Lazy behavior:** Simply calling the method `links` on the page object will not give us the list of links:

In [ ]:
page.links()

We need to iterate over this object to get the links, which themselves are objects pointing to the pages.

### 2.b What's in a page object?

As we saw above, each link shows up as a page object in the list of links. This is because these are all Wikipedia articles. Let's verify again that each page is an object:

In [ ]:
type(page)

We can use the Python built-in function `dir` to find out what properties or methods we can call on this object:

In [ ]:
onePage = links[10]
dir(onePage)

Let's try out some of the properties:

In [ ]:
# length of page in characters
onePage.length

In [ ]:
# name of the page
onePage.name

In [ ]:
# timestamp of when the page was "touched" or modified (not necessarily edited by users)
onePage.touched

**Note:** Notice the type `time.struct_time` that is used to represent time in Wikipedia. (There is a separate tutorial that explains this type.)

<a id="sec3"></a>

## 3. The Wellesley College page

Let's get the Wellesley College page and look at its properties. Instead of searching for "Wellesley", let's search for "Wellesley College":

In [ ]:
wcp = site.pages['Wellesley College']
wcp.name

Is it a protected page? Meaning, can anyone edit it, or are there some restrictions in place? In Wikipedia, some pages are protected to prevent vandalism.

In [ ]:
wcp.protection

No, it's not. But, we can easily find a page that is protected:

In [ ]:
hcp = site.pages['Hillary Clinton']
hcp.name

In [ ]:
hcp.protection

Notice that the result this time looks different from that of the Wellesley page. For example, only **autoconfirmed** users can edit the page. **autoconfirmed** means user accounts that have made at least 10 edits to the English Wikipedia and after 4 days have passed since the time of their first edit. You can learn more about levels of user access on Wikipedia [in this page](https://en.wikipedia.org/wiki/Wikipedia:User_access_levels).

What is the length of the page in characters?

In [ ]:
wcp.length # length of page in characters

Get external links from this page (links that go outside Wikipedia):

In [ ]:
extlinks = [el for el in wcp.extlinks()]
len(extlinks)

In [ ]:
extlinks[:10]

Find all Wikipedia pages that link to Wellesley College, these are known as **backlinks**:

In [ ]:
backlinks = [el for el in wcp.backlinks()]
len(backlinks)

That is a lot of backlinks that point to the Wellesley College page from other Wikipedia pages!

In [ ]:
backlinks[:10]

Finally, look at the links from this page to other Wikipedia pages:

In [ ]:
links = [el for el in wcp.links()]
len(links)

In [ ]:
links[:10]

We can say that more pages link to Wellesley College than vice-versa.

**IMPORTANT:** The links to other pages are useful to find things that are related. Even better are reciprocal links: pages that point to each-other.

<a id="sec4"></a>
## 4. Exploring the Wellesley page revisions

We can see that the object `page` has two properties, `revision` and `revisions`, let's look at them:

In [ ]:
wcp.revision

In [ ]:
wcp.revisions

The message shows that `revisions` is a method, not a property, we'll need parens to access it:

In [ ]:
wcp.revisions()

We can see the pattern now, most functions return **lazy objects**, because the user might not be interested in everything.  

**Get all revisions:** We can get all revisions we want by looping through the list iterator. This might take a few seconds.

In [ ]:
revisions = [rev for rev in wcp.revisions()]
len(revisions)

In [ ]:
revisions[:3]

### 4.a Find users in revisions

Each revision is stored as a Python dictionary, so we can easily extract the users:

In [ ]:
users = [rev['user'] for rev in revisions]

We'll use `Counter` to create a dict of users with their counts and then print these users based on the number of edits, with the most common edits at the top:

In [ ]:
from collections import Counter

usersDct = Counter(users)
usersDct.most_common(10)

**Note:** When I taught CS 234 in 2017, CS 234 students edited the Wellesley College page on Wikipedia. 

Let's check the count of edits for some of CS 234 editors of the page:

In [ ]:
usersDct['Imanh19']

In [ ]:
usersDct['Angelinahli']

How many unique users have edited this page?

In [ ]:
print(f"{len(usersDct)} unique users have edited {len(revisions)} times the {wcp.name} Wikipedia page.")

### 4.b Working with timestamps

Each revision contains a timestamp. Let's convert that to a datetime object to make it easier to work with it.  
**NOTE:** To make sense of this part, you need to have completed the notebook on working with date & time objects in Week 3 tasks.

In [ ]:
ts = revisions[0]['timestamp']
ts

In [ ]:
type(ts)

The following modules will work together to make the conversion from `timestruct` to `datetime`:

In [ ]:
from time import mktime
from datetime import datetime

# turn an object from type struct_time to datetime
datetime.fromtimestamp(mktime(ts))

Now that we have a datetime object, we can do many things:

1. group number of revisions by day
2. group number of revisions by month or year
3. group revisions by user revisions per day

A reminder that a datetime object has properties to access values such as year and month:

In [ ]:
dt = datetime.fromtimestamp(mktime(ts))
print(dt.year)
print(dt.month)
print(dt.day)

As well as a useful method to return only the date (without the time portion):

In [ ]:
print(dt.date())

This is especially useful in the succeeding example.

**Example: What was the day with most revisions?**

First, we convert all timestamps into string dates, just because it is easier to store them than datetime objects.

In [ ]:
def createDateTime(timestamp):
    """convert a timestruct to datetime"""
    return datetime.fromtimestamp(mktime(timestamp))

dates = [str(createDateTime(rev['timestamp']).date()) for rev in revisions]

# what does str(createDateTime(rev['timestamp']).date()) do?
# 1. it call the function createDateTime with each revision's timestamp object
# 2. then it applies the method date() on the returned datetime object, to get a date object
# 3. it converts the date object into a string

dates[:10]

**Find days with most edits**

We can do this in the same way we found the users with most edits, using the `Counter` constructor:

In [ ]:
datesDct = Counter(dates)
datesDct.most_common(10)

<a id='sec5'></a>
## 5. Wikipedia Pageviews

Similarly to the mwclient library we just saw, another library, mwviews, provides easy access to pageviews of every Wikipedia article.

First let's install the library:



In [ ]:
pip install mwviews

Now let's use this library to request pageviews:

In [ ]:
from mwviews.api import PageviewsClient

p = PageviewsClient(user_agent="CS315 Student learning about Wikipedia")

data = p.article_views('en.wikipedia', 
                       ['Wellesley College', 'Hillary Clinton'], # we can ask for multiple pages at once
                       start='20260101', end='20260131')

We should check first what kind of values are stored in the `data` variable:

In [ ]:
type(data)

It's a dictionary, so we can look up its items:

In [ ]:
list(data.items())[:3]

Now, let's try to look up some Kdramas from 2024, first let's make sure they have Wikipedia pages:

In [ ]:
kdramas = ["Queen of Tears", "Lovely Runner", "Welcome to Samdal-ri"]

for k in kdramas:
    p = site.pages[k]
    print(p)

Initializing and calling the pageview client:

In [ ]:
p = PageviewsClient(user_agent="Kdrama Lover (wendy_wellesley@gmail.com)")
data = p.article_views('en.wikipedia', kdramas, 
                       start='20260801', end='20260831')

list(data.items())[:5]

**Your Turn:** Get the views of these Chinese Dramas:

In [ ]:
cdramas = ['Pursuit of Jade', "Hidden Love", "When I Fly Towards You"]

# Your code here

### Saving results into a CSV

Let's see if we ask for the pageviews of a single drama for the entire lifetime of the drama. There are two ways to approach the issue of what is the "lifetime of the drama".

1. The day in which the Wiki page about the drama was created?
2. The day in which the drama started being broadcasted / streamed? 

We can get the day of the Wiki page through the revision history, or we can get the release of a drama on their MyDramaList. 

**Note**

For the purposes of saving a single file, I looked up the Wikipedia History to find the first edit on the page and this is how I got the data for Queen of Tears and Crash Landing on You. However, if we want to do this for hundreds of pages, we should use an automated process.

In [ ]:
import datetime
import pandas as pd


def export_dict_to_csv(data):
    try:
        # Grab the first key from the first non-empty inner dictionary
        title_key = next(k for inner in data.values() for k in inner)
    except StopIteration:
        print("Dictionary or inner data is empty.")
        return

    # Extract rows and save
    rows = [
        {"date": dt.strftime("%Y-%m-%d"), "count": inner.get(title_key)}
        for dt, inner in data.items()
    ]

    filename = f"{title_key}.csv"
    pd.DataFrame(rows).to_csv(filename, index=False)
    print(f"Saved to {filename}")


Save the data for Crash Landing on You:

In [ ]:
data = p.article_views('en.wikipedia', "Crash Landing on You", 
                       start='20190701', end='20260918')

export_dict_to_csv(data)

Save the data for Queen of Tears:

data = p.article_views('en.wikipedia', "Queen of Tears", 
                       start='20220401', end='20260918')

export_dict_to_csv(data)